In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install faker
import numpy as np
import openpyxl as openpyxl
import faker as fk
import pandas as pd
import random as random
import duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.8 MB/s eta 0:00:0000:0100:01


In [3]:
#creación de variables

n_pacientes = 10000
areas = ['UCI', 'Urgencias','Hospitalización General' ]
diagnosticos =['Neumonía','EPOC','Hipertensión','Infarto','Diabetes','Fractura','Apendicitis']

#Creación de DataFrame

hosp_data = {
    "id_paciente":random.choices(range(100000,120000),k=10000),
    "area":random.choices(areas,k=10000),
    "diagnostico":random.choices(diagnosticos ,k=10000),
    "readmision":np.random.choice(a=[0,1], size=10000, p=[0.85,0.15])
}


df_hosp = pd.DataFrame(hosp_data)


#Agregar columna de dias de estancia

dias_estancia = {
    'Neumonía':    (5, 14),
    'EPOC':        (4, 12),
    'Hipertensión':(2, 7),
    'Infarto':     (7, 21),
    'Diabetes':    (3, 10),
    'Fractura':    (3, 14),
    'Apendicitis': (1, 5)
}

df_hosp['dias_estancia']=df_hosp['diagnostico'].apply(
    lambda diag : random.randint(dias_estancia[diag][0],dias_estancia[diag][1])
)
    
df_hosp['Fecha_Ingreso'] = random.choices(pd.date_range('2024-01-01','2024-12-31'), k= 10000)
df_hosp.head()



,id_paciente,area,diagnostico,readmision,dias_estancia,Fecha_Ingreso
0,109554,Urgencias,Hipertensión,0,5,2024-10-11
1,100698,UCI,Neumonía,0,13,2024-06-28
2,109366,Urgencias,Neumonía,0,13,2024-05-01
3,118858,Urgencias,Neumonía,0,11,2024-12-16
4,114344,Urgencias,Neumonía,0,13,2024-08-27


In [7]:
con = duckdb.connect()
con.register('hospital',df_hosp)

#KPI de Ocupación por Área
kpi_ocupacion = con.execute('''
SELECT
    area,
    COUNT(*),
    ROUND(COUNT(area) * 100 / (SELECT COUNT(*) FROM hospital),1) AS porcentaje_area
FROM hospital
GROUP BY area
ORDER BY porcentaje_area DESC
'''
).df()

#KPI Tiempo Promedio por Diagnostico

kpi_estancia = con.execute("""
SELECT
    diagnostico,
    SUM(dias_estancia) AS total_tiempo_diag,
    COUNT(*) AS total_pac_diag,
    ROUND(SUM(dias_estancia)/COUNT(*),1) AS promedio_tiempo
FROM hospital
GROUP BY diagnostico
ORDER BY promedio_tiempo DESC
""").df()

#KPI Tasa de Readmisión por Diagnóstico

kpi_readmision = con.execute("""
SELECT
    diagnostico,
    SUM(readmision) AS readmision_diagnostico,
    COUNT(*) AS total_pacientes,
    ROUND(SUM(readmision)*100/COUNT(*),1) AS promedio_readmision
FROM hospital
GROUP BY diagnostico
ORDER BY promedio_readmision DESC
""").df()


In [14]:
#Exportar KPIs
kpi_ocupacion.to_csv('kpi_ocupacion.csv', index = False)
kpi_estancia.to_csv('kpi_estancia.csv', index = False)
kpi_readmision.to_csv('kpi_readmision.csv', index = False)